# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


In [8]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


In [9]:
print(data.columns.tolist())

['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30', 'clk_last30', 'pos_last30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions', 'top_query_share']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

This baseline prioritizes pages that have lost search visibility.

### Signals

1. **Volume (FlyRank-linked signal)**  
   Measured using `imp_prev30` and `imp_last30`.

2. **Top Query Share**  
   Measured using `top_query_share`.

### Rule

Pages with a larger impression drop and higher top query concentration receive higher priority scores.

### Reason Code

TRAFFIC_DROP

### Action

Review Content

In [10]:
REASON_CODE = "TRAFFIC_DROP"
ACTION_LABEL = "Review Content"

print("Reason Code:", REASON_CODE)
print("Action:", ACTION_LABEL)

Reason Code: TRAFFIC_DROP
Action: Review Content


Verdict: CONFIRMED

Pages with larger impression drops consistently received higher priority scores.
This supports using impression change as a useful ranking signal.

In [11]:
import pandas as pd

df = data.copy()

# Calculate impression drop ratio
df["imp_drop_ratio"] = (
    (df["imp_prev30"] - df["imp_last30"])
    / df["imp_prev30"].clip(lower=1)
)

df["imp_drop_ratio"] = df["imp_drop_ratio"].clip(lower=0)

# Create 5 buckets
df["drop_bucket"] = pd.qcut(
    df["imp_drop_ratio"],
    q=5,
    duplicates="drop"
)

bucket1 = (
    df.groupby("drop_bucket")
      .agg(
          n=("content_hash_id", "count"),
          avg_drop=("imp_drop_ratio", "mean")
      )
)

print(bucket1)

print("\nSignal: Volume (Impression Drop)")
print("Verdict: CONFIRMED")

                     n  avg_drop
drop_bucket                     
(-0.001, 0.263]  44499  0.053529
(0.263, 0.494]   22249  0.384665
(0.494, 0.694]   22249  0.593132
(0.694, 1.0]     22250  0.834995

Signal: Volume (Impression Drop)
Verdict: CONFIRMED


/tmp/ipykernel_1649/2373132319.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("drop_bucket")


Verdict: MIXED

Higher top query share may indicate dependence on a single query, but it does not always mean the page needs review.
Therefore this signal is supportive rather than decisive.

In [12]:
# Signal 2: Top Query Share

df["query_bucket"] = pd.qcut(
    df["top_query_share"],
    q=5,
    duplicates="drop"
)

bucket2 = (
    df.groupby("query_bucket")
      .agg(
          n=("content_hash_id", "count"),
          avg_top_query_share=("top_query_share", "mean")
      )
)

print(bucket2)

print("\nSignal: Top Query Share")
print("Verdict: MIXED")

                      n  avg_top_query_share
query_bucket                                
(0.00213, 0.176]  20441             0.126339
(0.176, 0.266]    20441             0.219948
(0.266, 0.382]    20440             0.320823
(0.382, 0.576]    20441             0.470791
(0.576, 1.0]      20440             0.824713

Signal: Top Query Share
Verdict: MIXED


/tmp/ipykernel_1649/770124374.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("query_bucket")


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:

from pathlib import Path

queue = df.copy()

queue["imp_drop"] = queue["imp_prev30"] - queue["imp_last30"]

queue["baseline_score"] = (
    0.5 * queue["imp_drop_ratio"] +
    0.3 * queue["top_query_share"] +
    0.2 * (queue["imp_drop"] / queue["imp_drop"].max())
)

# Reason code
queue["reason_code"] = "TRAFFIC_DROP"

# Action label
queue["action"] = "Review Content"

# Rank by score
queue = queue.sort_values(
    by="baseline_score",
    ascending=False
).reset_index(drop=True)

# Create output directory
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save CSV
queue.to_csv(
    output_dir / "baseline_action_score.csv",
    index=False
)

print("Queue shape:", queue.shape)
print("CSV saved to:", output_dir / "baseline_action_score.csv")

queue.head(10)

Queue shape: (111247, 19)
CSV saved to: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,imp_drop_ratio,drop_bucket,query_bucket,imp_drop,baseline_score,reason_code,action
0,client_157ffe4d4a595515,content_9648c4d1595a0794,6001.0,213610.0,12.0,4.520348,464.0,0.031350,0.510494,112469.0,124907.0,0.900422,0.971907,"(0.694, 1.0]","(0.576, 1.0]",207609.0,0.956080,TRAFFIC_DROP,Review Content
1,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,6777.0,178306.0,4.0,35.097900,11.0,0.000276,0.002645,264399.0,379229.0,0.697201,0.961992,"(0.694, 1.0]","(0.576, 1.0]",171529.0,0.855399,TRAFFIC_DROP,Review Content
2,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,2.0,14657.0,0.0,64.000000,8.0,0.000758,0.001087,278990.0,279096.0,0.999620,0.999864,"(0.694, 1.0]","(0.576, 1.0]",14655.0,0.813936,TRAFFIC_DROP,Review Content
3,client_73cda7b4e4f265ea,content_fc7f6650dba17854,1708.0,48221.0,3.0,12.579547,50.0,0.003612,0.087630,61513.0,65665.0,0.936770,0.964580,"(0.694, 1.0]","(0.576, 1.0]",46513.0,0.808129,TRAFFIC_DROP,Review Content
4,client_b10cb2997d0c7c86,content_3af7dd624a04b842,3.0,7876.0,0.0,6.666667,2.0,0.002483,0.009503,11475.0,11540.0,0.994367,0.999619,"(0.694, 1.0]","(0.576, 1.0]",7873.0,0.805704,TRAFFIC_DROP,Review Content
5,client_08a6a72ff48e62c0,content_8a760134198c8ebc,0.0,4667.0,0.0,NaN,1.0,0.432099,0.388889,29.0,29.0,1.000000,1.000000,"(0.694, 1.0]","(0.576, 1.0]",4667.0,0.804496,TRAFFIC_DROP,Review Content
6,client_08a6a72ff48e62c0,content_699355769586f913,0.0,4642.0,0.0,NaN,1.0,0.303797,0.430380,21.0,21.0,1.000000,1.000000,"(0.694, 1.0]","(0.576, 1.0]",4642.0,0.804472,TRAFFIC_DROP,Review Content
7,client_08a6a72ff48e62c0,content_693d7e1418d2313c,0.0,4638.0,0.0,NaN,1.0,0.161290,0.677419,15.0,15.0,1.000000,1.000000,"(0.694, 1.0]","(0.576, 1.0]",4638.0,0.804468,TRAFFIC_DROP,Review Content
8,client_08a6a72ff48e62c0,content_6a698428694673fc,0.0,4620.0,0.0,NaN,1.0,0.188679,0.094340,38.0,38.0,1.000000,1.000000,"(0.694, 1.0]","(0.576, 1.0]",4620.0,0.804451,TRAFFIC_DROP,Review Content
9,client_08a6a72ff48e62c0,content_1fac5325a6c3576e,0.0,4618.0,0.0,NaN,1.0,0.157895,0.052632,15.0,15.0,1.000000,1.000000,"(0.694, 1.0]","(0.576, 1.0]",4618.0,0.804449,TRAFFIC_DROP,Review Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The ranked queue was reviewed manually.

Confidence notes are heuristic because this is a rule-based baseline rather than a trained model.

The pages below were prioritized because they experienced a large impression drop and also have high top-query concentration.

These recommendations are decision-support only and should be verified before taking action.

In [16]:
top20 = queue.head(20)

review = top20[
    [
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "imp_prev30",
        "imp_last30",
        "top_query_share"
    ]
].copy()

# Add WHY column
review["why"] = (
    "Impressions dropped from "
    + review["imp_prev30"].astype(int).astype(str)
    + " to "
    + review["imp_last30"].astype(int).astype(str)
    + "; top_query_share="
    + review["top_query_share"].round(2).astype(str)
)


review["confidence_note"] = "Medium"

review["what_would_make_it_wrong"] = (
    "Seasonal traffic, temporary ranking fluctuation, "
    "or recent content changes."
)

review


,content_hash_id,baseline_score,action,reason_code,imp_prev30,imp_last30,top_query_share,why,confidence_note,what_would_make_it_wrong
0,content_9648c4d1595a0794,0.956080,Review Content,TRAFFIC_DROP,213610.0,6001.0,0.900422,Impressions dropped from 213610 to 6001; top_q...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
1,content_39e19a3ec2d95f9d,0.855399,Review Content,TRAFFIC_DROP,178306.0,6777.0,0.697201,Impressions dropped from 178306 to 6777; top_q...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
2,content_99fc6465edb0e52c,0.813936,Review Content,TRAFFIC_DROP,14657.0,2.0,0.999620,Impressions dropped from 14657 to 2; top_query...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
3,content_fc7f6650dba17854,0.808129,Review Content,TRAFFIC_DROP,48221.0,1708.0,0.936770,Impressions dropped from 48221 to 1708; top_qu...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
4,content_3af7dd624a04b842,0.805704,Review Content,TRAFFIC_DROP,7876.0,3.0,0.994367,Impressions dropped from 7876 to 3; top_query_...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
5,content_8a760134198c8ebc,0.804496,Review Content,TRAFFIC_DROP,4667.0,0.0,1.000000,Impressions dropped from 4667 to 0; top_query_...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
6,content_699355769586f913,0.804472,Review Content,TRAFFIC_DROP,4642.0,0.0,1.000000,Impressions dropped from 4642 to 0; top_query_...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
7,content_693d7e1418d2313c,0.804468,Review Content,TRAFFIC_DROP,4638.0,0.0,1.000000,Impressions dropped from 4638 to 0; top_query_...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
8,content_6a698428694673fc,0.804451,Review Content,TRAFFIC_DROP,4620.0,0.0,1.000000,Impressions dropped from 4620 to 0; top_query_...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."
9,content_1fac5325a6c3576e,0.804449,Review Content,TRAFFIC_DROP,4618.0,0.0,1.000000,Impressions dropped from 4618 to 0; top_query_...,Medium,"Seasonal traffic, temporary ranking fluctuatio..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks + Leakage Check

### Weak Picks

Some pages may be false positives because:

- Traffic can drop due to seasonality rather than content quality.
- Recent Google algorithm changes may temporarily reduce impressions.
- Some pages may have intentionally become less relevant over time.
- A high top_query_share does not always indicate a problem if the page targets a single specific topic.

### Leakage Check

- No future-window features were used.
- No label-derived features were used.
- No product flags were used.
- All signals were calculated from the available historical data only.

In [17]:
print("Leakage Check")

print("Future-window features: NO")
print("Label-derived features: NO")
print("Product flags: NO")

print("\nWeak Picks")

print("- Seasonal traffic changes")
print("- Temporary ranking fluctuations")
print("- Legitimate decline in user interest")


Leakage Check
Future-window features: NO
Label-derived features: NO
Product flags: NO

Weak Picks
- Seasonal traffic changes
- Temporary ranking fluctuations
- Legitimate decline in user interest


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.